In [1]:
!pip install fastapi uvicorn > /dev/null

import uvicorn
import threading
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  #  Замените на адрес сервиса анализа, если знаете
    allow_methods=["*"],
    allow_headers=["*"],
    allow_credentials=True,
)

@app.get("/")
async def root():
    return {"message": "Hello World"}

@app.post("/test")
async def test():
    return {
        "status": "ok"
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server)
server_thread.start()

INFO:     Started server process [38]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     89.111.27.213:0 - "GET / HTTP/1.1" 200 OK
INFO:     89.111.27.213:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


In [2]:
import os
import subprocess
import threading
import time
import socket
import urllib

def run_command(cmd, printout=False):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                        stderr=subprocess.PIPE, text=True, bufsize=1)
    stdout, stderr = p.communicate()
    if printout:
        if stdout:
            print(stdout)
        if stderr:
            print(stderr)
    return p

# Tunnel

In [3]:
from abc import ABC, abstractmethod
import threading

class Tunnel(ABC):
    def __init__(self, token: str | None =None):
        self.token = token

    @abstractmethod
    def install(self):
        pass

    @abstractmethod
    def thread(self, port=8000):
        pass

    def start(self):
        threading.Thread(target=self.thread, daemon=True).start()

## Zrok

In [4]:
class ZROK(Tunnel):
    name = "zrok"

    def install(self):
        # os.chdir('/tmp')
        get_ipython().system('wget -q https://github.com/openziti/zrok/releases/download/v2.0.0-rc7/zrok_2.0.0-rc7_linux_amd64.tar.gz')
        get_ipython().system('tar -xvzf zrok_2.0.0-rc7_linux_amd64.tar.gz')
        get_ipython().system('chmod +x zrok2')
        print("🔐 Enabling zrok environment...")
        cmd = f"./zrok2 enable {self.token}"
        p = run_command(cmd)
        if p.returncode == 0:
            print("✅ zrok enabled successfully!")
        else:
            if 'you already have an enabled environment' in p.stderr:
                print("✅ zrok is already enabled!")
            else:
                print(f"❌ Error enabling zrok: {p.stderr}")

    def thread(self, port=8000):
        cmd = f"./zrok2 share public localhost:{port} --headless"
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
        print('====== zrok enable =====')
        for line in p.stdout:
            print('[ZROK]:', line.strip())
            if 'shares.zrok.io' in line:
                import re
        
                m = re.search(r'(?im)\b([a-z0-9-]+\.shares\.zrok\.io)\b', line)
                url = m.group(1) if m else None
                
                msg = f'Zrok URL: {url}'
                border = '        +' + '-' * (len(msg) + 2) + '+'
                print()
                print(border)
                print(f".       | {msg} |")
                print(border)
                print()
            if '[ERROR]' in line:
                msg = 'Zrok Error: Cannot create public link. Visit https://api.zrok.io and delete some environments.'
                border = '        +' + '-' * (len(msg) + 2) + '+'
                print(border)
                print(f".       | {msg} |")
                print(border)

## Ngrok

In [5]:
class NGROK(Tunnel):
    name = "ngrok"

    def install(self):
        get_ipython().system('pip install -q pyngrok')
        print("pyngrok installed!")

    def thread(self, port=8000):
        print("\n Attention! Ngrok requires VPN in Russia!")
        from pyngrok import ngrok
        ngrok.set_auth_token(self.token)
        http_tunnel = ngrok.connect(port)
        print(f'NGROK URL: {http_tunnel.public_url}')

## localtunnel

In [6]:
class LocalTunnel(Tunnel):
    name = "localtunnel"

    def install(self):
        get_ipython().system('npm install -g localtunnel')
        print("localtunnel installed!")
    
    def thread(self, port=8000):
        print("Tunnel Password:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
        p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
        for line in p.stdout:
            print(line.decode(), end='')

## cloudflared

In [7]:
class Cloudflared(Tunnel):
    name = "cloudflared"

    def install(self):
        get_ipython().system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb')
        get_ipython().system('dpkg -i cloudflared-linux-amd64.deb')
        print("cloudflared installed!")
        
    def thread(self, port=8000):
        get_ipython().system(f'cloudflared tunnel --url http://localhost:{port}')

## pinggy

In [15]:
class Pinggy(Tunnel):
    name = "pinggy"

    def install(self):
        pass
        
    def thread(self, port=8000):
        print("\n🌐 Starting Pinggy tunnel...")
        print("\n Caution! Pinggy requires VPN in Russia!")
        print("⏳ Waiting for Pinggy URL...\n")
        
        # Создаем SSH конфиг
        ssh_config = """
    Host pinggy
        HostName a.pinggy.io
        Port 443
        StrictHostKeyChecking no
        UserKnownHostsFile /dev/null
        ServerAliveInterval 30
        """
        
        get_ipython().system('mkdir -p ~/.ssh')
        with open(os.path.expanduser('~/.ssh/config'), 'w') as f:
            f.write(ssh_config)
        
        # Запуск pinggy
        cmd = f'ssh -R0:localhost:{port} pinggy'
        
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, 
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
        
        url_found = False
        for line in p.stdout:
            print('[PINGGY]:', line.strip())
            
            if not url_found and ('http://' in line or 'https://' in line):
                import re
                # Ищем URL формата http://xxxxx.a.pinggy.link
                urls = re.findall(r'https?://[a-zA-Z0-9\-\.]+\.pinggy\.[a-z]+', line)
                if urls:
                    url = urls[0]
                    msg = f'Pinggy URL: {url}'
                    border = '        +' + '-' * (len(msg) + 2) + '+'
                    print()
                    print(border)
                    print(f".       | {msg} |")
                    print(border)
                    print()
                    url_found = True

# Test

In [16]:
tunnel_builders: dict[str, Tunnel] = {
    "zrok": (ZROK, "dGi5n5cyC8q7"),
    "ngrok": (NGROK, "2a39vbvEM8fO1aj5h7p5VuibQvK_3SACZDUBhgmWb9cJcUqmA"),
    "localtunnel": (LocalTunnel, None),
    "cloudflared": (Cloudflared, None),
    "pinggy": (Pinggy, None),
}

In [18]:
CONNECTION = "pinggy"

print("CONNECTION:", CONNECTION)

tunnel_builder, token = tunnel_builders[CONNECTION]
tunnel = tunnel_builder(token=token)

tunnel.install()
tunnel.start()

CONNECTION: pinggy

🌐 Starting Pinggy tunnel...

 Caution! Pinggy requires VPN in Russia!
⏳ Waiting for Pinggy URL...

[PINGGY]: Pseudo-terminal will not be allocated because stdin is not a terminal.
[PINGGY]: Warning: Permanently added '[a.pinggy.io]:443,[141.94.45.153]:443' (RSA) to the list of known hosts.
[PINGGY]: Allocated port 4 for remote forward to localhost:8000
[PINGGY]: You are not authenticated.
[PINGGY]: Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io

        +-----------------------------------------+
.       | Pinggy URL: https://dashboard.pinggy.io |
        +-----------------------------------------+

[PINGGY]: http://hqsal-34-7-18-124.a.free.pinggy.link
[PINGGY]: https://hqsal-34-7-18-124.a.free.pinggy.link
[PINGGY]: 
[PINGGY]: RB: 0, SB: 0, TC: 0, AC: 0
[PINGGY]: RB: 0, SB: 0, TC: 0, AC: 0
[PINGGY]: RB: 967, SB: 150, TC: 1, AC: 1
[PINGGY]: RB: 967, SB: 150, TC: 1, AC: 1
[PINGGY]: RB: 967, SB: 150

In [11]:
# CONNECTION: localtunnel
# m#################.] / reify:yargs-parser: http fetch GET 200 https://registry.
# added 22 packages in 2s

# 3 packages are looking for funding
#   run `npm fund` for details
# npm notice 
# npm notice New major version of npm available! 10.5.0 -> 11.11.0
# npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.11.0
# npm notice Run npm install -g npm@11.11.0 to update!
# npm notice 
# localtunnel installed!
# Tunnel Password: 34.141.170.17
# your url is: https://dull-animals-double.loca.lt

In [12]:
# CONNECTION: cloudflared
# Selecting previously unselected package cloudflared.
# (Reading database ... 116071 files and directories currently installed.)
# Preparing to unpack cloudflared-linux-amd64.deb ...
# Unpacking cloudflared (2026.3.0) ...
# Setting up cloudflared (2026.3.0) ...
# Processing triggers for man-db (2.9.1-1) ...
# cloudflared installed!
# 2026-03-11T10:48:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
# 2026-03-11T10:48:41Z INF Requesting new quick Tunnel on trycloudflare.com...
# 2026-03-11T10:48:45Z INF +--------------------------------------------------------------------------------------------+
# 2026-03-11T10:48:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
# 2026-03-11T10:48:45Z INF |  https://imports-hearings-miracle-discuss.trycloudflare.com                                |
# 2026-03-11T10:48:45Z INF +--------------------------------------------------------------------------------------------+
# 2026-03-11T10:48:45Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
# 2026-03-11T10:48:45Z INF Version 2026.3.0 (Checksum 4a9e50e6d6d798e90fcd01933151a90bf7edd99a0a55c28ad18f2e16263a5c30)
# 2026-03-11T10:48:45Z INF GOOS: linux, GOVersion: go1.24.13, GoArch: amd64
# 2026-03-11T10:48:45Z INF Settings: map[ha-connections:1 protocol:quic url:http://localhost:8000]
# 2026-03-11T10:48:45Z INF cloudflared will not automatically update if installed by a package manager.
# 2026-03-11T10:48:45Z INF Generated Connector ID: 370a277d-4739-4144-a33c-fb0b6952c55e
# 2026-03-11T10:48:45Z INF Initial protocol quic
# 2026-03-11T10:48:45Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
# 2026-03-11T10:48:45Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
# 2026-03-11T10:48:45Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
# 2026-03-11T10:48:45Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
# 2026-03-11T10:48:45Z INF Starting metrics server on 127.0.0.1:20241/metrics
# 2026-03-11T10:48:46Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveP256] connIndex=0 event=0 ip=198.41.200.33
# 2026/03/11 10:48:46 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
# 2026-03-11T10:48:46Z INF Registered tunnel connection connIndex=0 connection=cca74685-b63f-4031-b0e8-743113d6f95d event=0 ip=198.41.200.33 location=ams01 protocol=quic

In [ ]:
# CONNECTION: pinggy

# 🌐 Starting Pinggy tunnel...

#  Caution! Pinggy requires VPN in Russia!
# ⏳ Waiting for Pinggy URL...

# [PINGGY]: Pseudo-terminal will not be allocated because stdin is not a terminal.
# [PINGGY]: Warning: Permanently added '[a.pinggy.io]:443,[141.94.45.153]:443' (RSA) to the list of known hosts.
# [PINGGY]: Allocated port 4 for remote forward to localhost:8000
# [PINGGY]: You are not authenticated.
# [PINGGY]: Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io

#         +-----------------------------------------+
# .       | Pinggy URL: https://dashboard.pinggy.io |
#         +-----------------------------------------+

# [PINGGY]: http://hqsal-34-7-18-124.a.free.pinggy.link
# [PINGGY]: https://hqsal-34-7-18-124.a.free.pinggy.link